In [21]:
import pandas as pd
import os
import numpy as np
from utils.index import (
    cat_cols_none_impute,
    cols_zero_impute,
    fill_categorical_nulls,
    fill_zero,
    outliers,
    prep_data_numeric_cols,
)

train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

# Remove outliers from training set before merging to avoid affecting test stats
train_df = train_df.drop(outliers).reset_index(drop=True)

# Save the target and the length of train_df for splitting later
y_train = np.log(train_df["SalePrice"])
train_len = len(train_df)

# Drop target from train so we can concat
X_train_temp = train_df.drop(columns=["SalePrice", "Id"])
X_test_temp = test_df.copy().drop(columns=["Id"])

# 2. Merge DataFrames
all_data = pd.concat([X_train_temp, X_test_temp], axis=0).reset_index(drop=True)

# 3. Apply Transformations to the combined dataset
all_data = fill_categorical_nulls(all_data, cat_cols_none_impute, fill_value="None")
all_data = fill_zero(all_data, cols_zero_impute)
all_data = all_data.drop(columns=["Id"], errors="ignore")

# Median LotFrontage by Neighborhood across the whole set
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

all_data["Electrical"] = all_data["Electrical"].fillna(all_data["Electrical"].mode()[0])

# 4. Numeric Scaling
numeric_cols = all_data.select_dtypes(include=["int64", "float64"]).columns

# Process numeric columns (assuming this function handles scaling and returns the scaler)
# Note: For strict ML, you'd usually fit the scaler on the train portion only.
all_data_num, _, cat_cols = prep_data_numeric_cols(all_data, numeric_cols)

for col in cat_cols:
    all_data[col] = all_data[col].fillna("missing").astype(str)

# 6. Combine Features
X_all_final = pd.concat([all_data_num, all_data[cat_cols]], axis=1)

# 7. Split back into Train and Test
X_train_final = X_all_final.iloc[:train_len, :]
X_test_final = X_all_final.iloc[train_len:, :]

# If test_df had a target column (like in a local validation set), split it here:
# y_test = test_df["SalePrice"] if "SalePrice" in test_df.columns else None

id_col = test_df["Id"].copy()

Detected 21 skewed columns to fix.

FIXED [MSSubClass]: Positive Skew (1.37) -> Applied Log1p
FIXED [LotFrontage]: Positive Skew (1.10) -> Applied Log1p
FIXED [LotArea]: Positive Skew (13.29) -> Applied Log1p
FIXED [MasVnrArea]: Positive Skew (2.57) -> Applied Log1p
FIXED [BsmtFinSF1]: Positive Skew (0.97) -> Applied Log1p
FIXED [BsmtFinSF2]: Positive Skew (4.15) -> Applied Log1p
FIXED [BsmtUnfSF]: Positive Skew (0.92) -> Applied Log1p
FIXED [1stFlrSF]: Positive Skew (1.28) -> Applied Log1p
FIXED [2ndFlrSF]: Positive Skew (0.84) -> Applied Log1p
FIXED [LowQualFinSF]: Positive Skew (12.39) -> Applied Log1p
FIXED [GrLivArea]: Positive Skew (1.00) -> Applied Log1p
FIXED [BsmtHalfBath]: Positive Skew (3.94) -> Applied Log1p
FIXED [KitchenAbvGr]: Positive Skew (4.29) -> Applied Log1p
FIXED [GarageYrBlt]: Negative Skew (-3.90) -> Applied Yeo-Johnson
FIXED [WoodDeckSF]: Positive Skew (1.86) -> Applied Log1p
FIXED [OpenPorchSF]: Positive Skew (2.56) -> Applied Log1p
FIXED [EnclosedPorch]: Posi

In [3]:
from sklearn.model_selection import KFold, cross_val_score
import numpy as np


def rmse_cv(model, X, y):
    # Use K-Fold (k=10) for cross-validation
    kf = KFold(n_splits=10, shuffle=True, random_state=42).get_n_splits(X.values)

    # Calculate scores using cross-validation
    # Negative MSE is used as scoring, and the square root is taken for RMSE.
    rmse = np.sqrt(
        -cross_val_score(model, X.values, y, scoring="neg_mean_squared_error", cv=kf)
    )

    return rmse

In [7]:
from catboost import CatBoostRegressor


cat_feature_indices = [X_train_final.columns.get_loc(col) for col in cat_cols]

params = {
    "task_type": "GPU",
    "iterations": 1000,
    "learning_rate": 0.02,  # Very slow learning
    "depth": 4,  # Simple trees
    "l2_leaf_reg": 10,  # High penalty for complex leaves
    "random_strength": 2,  # Adds noise to split selection
    "min_data_in_leaf": 15,  # Prevents leaves from capturing single outliers
    "bagging_temperature": 1,
    "early_stopping_rounds": 50,
    "verbose": 200,
}

model = CatBoostRegressor(**params, cat_features=cat_feature_indices)

catboost_rmse = rmse_cv(model, X_train_final, y_train)

0:	learn: 0.3804500	total: 102ms	remaining: 1m 42s
200:	learn: 0.1418968	total: 12s	remaining: 47.6s
400:	learn: 0.1216027	total: 24.1s	remaining: 36s
600:	learn: 0.1124691	total: 35.1s	remaining: 23.3s
800:	learn: 0.1077534	total: 46.5s	remaining: 11.6s
999:	learn: 0.1047793	total: 58.2s	remaining: 0us
0:	learn: 0.3804040	total: 44.5ms	remaining: 44.4s
200:	learn: 0.1421073	total: 11.8s	remaining: 46.8s
400:	learn: 0.1232405	total: 23.5s	remaining: 35.1s
600:	learn: 0.1139195	total: 35.1s	remaining: 23.3s
800:	learn: 0.1089872	total: 46.8s	remaining: 11.6s
999:	learn: 0.1059194	total: 58.4s	remaining: 0us
0:	learn: 0.3749311	total: 63.7ms	remaining: 1m 3s
200:	learn: 0.1414605	total: 12s	remaining: 47.6s
400:	learn: 0.1211523	total: 24.1s	remaining: 36.1s
600:	learn: 0.1129725	total: 36.6s	remaining: 24.3s
800:	learn: 0.1085942	total: 48.6s	remaining: 12.1s
999:	learn: 0.1054442	total: 59.9s	remaining: 0us
0:	learn: 0.3764499	total: 45.9ms	remaining: 45.9s
200:	learn: 0.1377163	total:

In [8]:
print(f"CatBoost RMSE (Cross-Validation Mean): {catboost_rmse.mean():.4f}") 
print(f"CatBoost RMSE (Standard Deviation): {catboost_rmse.std():.4f}") 

CatBoost RMSE (Cross-Validation Mean): 0.1212
CatBoost RMSE (Standard Deviation): 0.0186


In [22]:
params = {
    "task_type": "GPU",
    "iterations": 2000,
    "learning_rate": 0.02,  # Very slow learning
    "depth": 5,  # Simple trees
    "l2_leaf_reg": 10,  # High penalty for complex leaves
    "random_strength": 2,  # Adds noise to split selection
    "min_data_in_leaf": 15,  # Prevents leaves from capturing single outliers
    "bagging_temperature": 1,
    "early_stopping_rounds": 50,
    "verbose": 200,
}

model = CatBoostRegressor(**params, cat_features=cat_feature_indices)

model.fit(X_train_final, y_train)

0:	learn: 0.3797771	total: 80ms	remaining: 2m 39s
200:	learn: 0.1382490	total: 15.5s	remaining: 2m 18s
400:	learn: 0.1184131	total: 30s	remaining: 1m 59s
600:	learn: 0.1088054	total: 45.2s	remaining: 1m 45s
800:	learn: 0.1032874	total: 1m	remaining: 1m 30s
1000:	learn: 0.0997730	total: 1m 15s	remaining: 1m 14s
1200:	learn: 0.0970090	total: 1m 30s	remaining: 59.9s
1400:	learn: 0.0949231	total: 1m 44s	remaining: 44.9s
1600:	learn: 0.0926655	total: 1m 59s	remaining: 29.7s
1800:	learn: 0.0908820	total: 2m 13s	remaining: 14.7s
1999:	learn: 0.0891531	total: 2m 27s	remaining: 0us


In [20]:
X_test_final.columns[51]

# mode_val = X_train_final["Utilities"].mode()[0]

# X_test_final["Utilities"] = X_test_final["Utilities"].fillna(mode_val).astype(str)

'Exterior1st'

In [23]:
def export_submission(model, id_col):
    preds = model.predict(X_test_final)

    submission = pd.DataFrame({"Id": id_col, "SalePrice": np.exp(preds)})

    model_name = type(model).__name__
    submission.to_csv(f"{model_name}_submission.csv", index=False)


export_submission(model, id_col)